# 03 — Quantize & Benchmark All Methods

Apply GPTQ, AWQ, BnB-NF4, and BnB-INT8 to the **merged** fine-tuned
`gemma4-e2b-medical-merged` model from notebook 02, then benchmark every
variant on the same metrics.

### Compatibility notes for Gemma 4 E2B
| Method | Status | Notes |
|--------|--------|-------|
| BnB NF4 (4-bit) | ✅ Works | dtype-level, model-agnostic |
| BnB INT8 (8-bit) | ✅ Works | dtype-level, model-agnostic |
| GPTQ (3/4/8-bit) | ⚠️ Try/Except | Needs `optimum` support for `gemma4` arch |
| AWQ (4-bit) | ⚠️ Try/Except | Needs `autoawq` support for `gemma4` arch |

GPTQ and AWQ cells are wrapped in `try/except` and will skip gracefully if
the library doesn't yet recognise the Gemma 4 architecture. BnB cells always
run and provide the primary quantization results.

**Hardware target**: Kaggle 2×T4 (16 GB each).

In [ ]:
!pip install -q "transformers>=4.50.0" peft bitsandbytes auto-gptq autoawq \
    datasets accelerate sentencepiece protobuf pandas tqdm

import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import gc
import time
import torch
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GPTQConfig,
)

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB")

# Path to the merged fine-tuned model produced by notebook 02
MERGED_MODEL_PATH = "gemma4-e2b-medical-merged"
RESULTS_FILE = "results/tables/all_results.csv"
os.makedirs("results/tables", exist_ok=True)

if not Path(MERGED_MODEL_PATH).exists():
    raise FileNotFoundError(
        f"{MERGED_MODEL_PATH} not found. Run notebook 02 first to produce the merged model."
    )

## Helper: Evaluation Function

Same benchmark suite as notebooks 01 and 02: WikiText-2 perplexity,
medical-text perplexity, PubMedQA accuracy, tokens/sec, peak VRAM.

In [ ]:
BATCH_SIZE = 2


def evaluate_model(model, tokenizer, model_name, device="cuda"):
    """Run full evaluation suite; returns results dict."""
    results = {"model": model_name}
    model.eval()

    # [1/4] WikiText-2 perplexity
    print(f"  [1/4] WikiText-2 perplexity...")
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:256]
    total_loss, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE), leave=False):
        batch = wiki_texts[i : i + BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors="pt", truncation=True, max_length=512, padding=True
        ).to(device)
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        n = enc["attention_mask"].sum().item()
        total_loss += out.loss.item() * n
        total_tokens += n
        del enc, out
    torch.cuda.empty_cache()
    results["perplexity_wikitext"] = torch.exp(
        torch.tensor(total_loss / total_tokens)
    ).item()
    print(f"    -> {results['perplexity_wikitext']:.2f}")

    # [2/4] Medical text perplexity
    print(f"  [2/4] Medical text perplexity...")
    pqa = load_dataset("pubmed_qa", "pqa_labeled", split="train")
    med_texts = []
    for ex in pqa:
        ctx_data = ex.get("context", {})
        ctxs = ctx_data.get("contexts", [])
        ctx = " ".join(ctxs) if isinstance(ctxs, list) else str(ctxs)
        if len(ctx.strip()) > 50:
            med_texts.append(ctx)
        if len(med_texts) >= 256:
            break
    total_loss, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(med_texts), BATCH_SIZE), leave=False):
        batch = med_texts[i : i + BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors="pt", truncation=True, max_length=512, padding=True
        ).to(device)
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        n = enc["attention_mask"].sum().item()
        total_loss += out.loss.item() * n
        total_tokens += n
        del enc, out
    torch.cuda.empty_cache()
    results["perplexity_medical"] = torch.exp(
        torch.tensor(total_loss / total_tokens)
    ).item()
    print(f"    -> {results['perplexity_medical']:.2f}")

    # [3/4] PubMedQA accuracy
    print(f"  [3/4] PubMedQA accuracy...")
    correct, total = 0, 0
    for ex in tqdm(pqa, total=200, leave=False):
        if total >= 200:
            break
        ctx_data = ex.get("context", {})
        ctxs = ctx_data.get("contexts", [])
        context = " ".join(ctxs) if isinstance(ctxs, list) else str(ctxs)
        prompt = (
            f"<start_of_turn>user\n"
            f"Context: {context}\n\n"
            f"Question: {ex['question']}\n"
            f"Answer with exactly one word: yes, no, or maybe.<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=768
        ).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        resp = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip().lower()
        pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
        gold = ex["final_decision"].lower().strip()
        if pred in ("yes", "no", "maybe") and pred == gold:
            correct += 1
        total += 1
        del inputs, out
        torch.cuda.empty_cache()
    results["pubmedqa_accuracy"] = correct / total if total else 0
    print(f"    -> {results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

    # [4/4] Inference speed + peak VRAM
    print(f"  [4/4] Inference speed & VRAM...")
    prompt = "<start_of_turn>user\nWhat is diabetes?<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=50, do_sample=False)
    torch.cuda.reset_peak_memory_stats()
    total_tok, total_t = 0, 0.0
    for _ in range(5):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
        torch.cuda.synchronize()
        total_tok += out.shape[1] - inputs["input_ids"].shape[1]
        total_t += time.perf_counter() - t0
    results["tokens_per_sec"] = total_tok / total_t
    results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / 1024**3
    print(f"    -> {results['tokens_per_sec']:.1f} tok/s")
    print(f"    -> {results['peak_vram_gb']:.2f} GB peak VRAM")

    return results


def save_results(results_dict):
    """Append/update row in all_results.csv (deduplicates by model name)."""
    df_new = pd.DataFrame([results_dict])
    if os.path.exists(RESULTS_FILE):
        df = pd.read_csv(RESULTS_FILE)
        df = df[df["model"] != results_dict["model"]]
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(RESULTS_FILE, index=False)
    return df

## Calibration Data (medical text from PubMedQA)

Using 64 medical PubMedQA context passages as GPTQ/AWQ calibration data.
This is intentional — the research question asks whether medical calibration
data changes which quantization method best preserves model quality.

In [ ]:
pqa_cal = load_dataset("pubmed_qa", "pqa_labeled", split="train")
calibration_texts = []
for ex in pqa_cal:
    ctx_data = ex.get("context", {})
    ctxs = ctx_data.get("contexts", [])
    ctx = " ".join(ctxs) if isinstance(ctxs, list) else str(ctxs)
    if len(ctx.strip()) > 50:
        calibration_texts.append(ctx)
    if len(calibration_texts) >= 64:
        break
print(f"Calibration samples: {len(calibration_texts)}")

## 1. GPTQ Quantization (8-bit, 4-bit, 3-bit)

> **Note**: GPTQ requires `optimum` to register the model's architecture.
> If `gemma4` is not yet in `optimum`'s supported list, these cells will
> print a warning and skip — BnB variants in section 3 & 4 will still run.

In [ ]:
# ── GPTQ 8-bit ─────────────────────────────────────────────────────────────
print("=" * 60)
print("GPTQ 8-bit")
print("=" * 60)
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    gptq_config = GPTQConfig(
        bits=8,
        group_size=128,
        desc_act=True,
        dataset=calibration_texts,
        tokenizer=tokenizer,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH,
        quantization_config=gptq_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    res = evaluate_model(model, tokenizer, "gemma4-e2b-med-gptq-8bit")
    save_results(res)
except Exception as e:
    print(f"[SKIP] GPTQ 8-bit failed: {e}")
    print("       This is expected if 'optimum' does not yet support gemma4 arch.")
finally:
    try:
        del model
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# ── GPTQ 4-bit ─────────────────────────────────────────────────────────────
print("=" * 60)
print("GPTQ 4-bit")
print("=" * 60)
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    gptq_config = GPTQConfig(
        bits=4,
        group_size=128,
        desc_act=True,
        dataset=calibration_texts,
        tokenizer=tokenizer,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH,
        quantization_config=gptq_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    res = evaluate_model(model, tokenizer, "gemma4-e2b-med-gptq-4bit")
    save_results(res)
except Exception as e:
    print(f"[SKIP] GPTQ 4-bit failed: {e}")
    print("       This is expected if 'optimum' does not yet support gemma4 arch.")
finally:
    try:
        del model
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# ── GPTQ 3-bit ─────────────────────────────────────────────────────────────
print("=" * 60)
print("GPTQ 3-bit")
print("=" * 60)
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    gptq_config = GPTQConfig(
        bits=3,
        group_size=128,
        desc_act=True,
        dataset=calibration_texts,
        tokenizer=tokenizer,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH,
        quantization_config=gptq_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    res = evaluate_model(model, tokenizer, "gemma4-e2b-med-gptq-3bit")
    save_results(res)
except Exception as e:
    print(f"[SKIP] GPTQ 3-bit failed: {e}")
    print("       This is expected if 'optimum' does not yet support gemma4 arch.")
finally:
    try:
        del model
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

## 2. AWQ Quantization (4-bit)

> **Note**: AWQ requires `autoawq` to support the model architecture.
> If `gemma4` is not registered, this cell skips gracefully.

In [ ]:
# ── AWQ 4-bit ──────────────────────────────────────────────────────────────
print("=" * 60)
print("AWQ 4-bit")
print("=" * 60)
try:
    from awq import AutoAWQForCausalLM

    awq_model = AutoAWQForCausalLM.from_pretrained(MERGED_MODEL_PATH)
    awq_tok = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)

    quant_config = {
        "zero_point": True,
        "q_group_size": 128,
        "w_bit": 4,
        "version": "GEMM",
    }
    awq_model.quantize(
        awq_tok, quant_config=quant_config, calib_data=calibration_texts
    )

    # Save quantised model so we can reload with AutoModelForCausalLM
    os.makedirs("models/quantized/awq-4bit", exist_ok=True)
    awq_model.save_quantized("models/quantized/awq-4bit")
    awq_tok.save_pretrained("models/quantized/awq-4bit")
    del awq_model
    torch.cuda.empty_cache()
    gc.collect()

    # Reload for evaluation
    model = AutoModelForCausalLM.from_pretrained(
        "models/quantized/awq-4bit", device_map="auto", torch_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained("models/quantized/awq-4bit")
    res = evaluate_model(model, tokenizer, "gemma4-e2b-med-awq-4bit")
    save_results(res)
except Exception as e:
    print(f"[SKIP] AWQ 4-bit failed: {e}")
    print("       This is expected if 'autoawq' does not yet support gemma4 arch.")
finally:
    try:
        del model
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

## 3. BitsAndBytes NF4 (4-bit)

Runtime quantization — no saved checkpoint; model is quantised on load.
Works with any model architecture supported by transformers.

In [ ]:
print("=" * 60)
print("BnB NF4 (4-bit)")
print("=" * 60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

res = evaluate_model(model, tokenizer, "gemma4-e2b-med-bnb-nf4")
save_results(res)
del model
torch.cuda.empty_cache()
gc.collect()

## 4. BitsAndBytes INT8 (8-bit)

LLM.int8 mixed-precision decomposition. Handles outlier features in fp16
and compresses the remaining weights to int8.

In [ ]:
print("=" * 60)
print("BnB INT8 (8-bit)")
print("=" * 60)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

res = evaluate_model(model, tokenizer, "gemma4-e2b-med-bnb-int8")
save_results(res)
del model
torch.cuda.empty_cache()
gc.collect()

## 5. All Results

In [ ]:
df = pd.read_csv(RESULTS_FILE)
print(f"\nResults file: {RESULTS_FILE}")
print(f"Rows: {len(df)}")
print("\nAll results:")
print("=" * 100)
df